In [ ]:
import os
import mlflow

os.environ["MLFLOW_TRACKING_USERNAME"] = "Roy7721"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "bc912b7d58bd2aca05abdc192e010414493a3886"

mlflow.set_tracking_uri("https://dagshub.com/Roy7721/yt_comment_analysis.mlflow")

# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import ADASYN
import mlflow.sklearn
import optuna

In [ ]:
# Step 1: Load and clean data
df = pd.read_csv('dataset.csv').dropna()
df = df.dropna(subset=['category'])

# Step 2: Split FIRST on raw text to avoid data leakage
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# Step 3: TF-IDF vectorizer - FIT ONLY on training data
ngram_range = (1, 2)  # Bigram
max_features = 10000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train = vectorizer.fit_transform(X_train_raw)  # Fit ONLY on training data
X_test = vectorizer.transform(X_test_raw)  # Transform test data (don't refit)

# Step 4: Apply ADASYN ONLY to training data to handle class imbalance
adasyn = ADASYN(random_state=42)
X_train_resampled, y_train_resampled = adasyn.fit_resample(X_train, y_train)

# Test data stays untouched for fair evaluation
X_train = X_train_resampled
y_train = y_train_resampled

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type and experiment info
        mlflow.set_tag("mlflow.runName", f"{model_name}_ADASYN_TFIDF_Bigrams_NoLeakage")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.set_tag("data_handling", "no_leakage_fixed")
        mlflow.set_tag("model_type", "MultinomialNB")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)
        mlflow.log_param("vectorizer_type", "TfidfVectorizer")
        mlflow.log_param("ngram_range", str(ngram_range))
        mlflow.log_param("vectorizer_max_features", max_features)
        mlflow.log_param("imbalance_handling", "ADASYN_on_train_only")
        mlflow.log_param("alpha", model.alpha)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report metrics
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")
        
        print(f"Run logged successfully for {model_name}")


# Step 5: Optuna objective function for Multinomial Naive Bayes
def objective_mnb(trial):
    alpha = trial.suggest_float('alpha', 1e-4, 1.0, log=True)

    model = MultinomialNB(alpha=alpha)
    model.fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))


# Step 6: Run Optuna for Multinomial Naive Bayes, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_mnb, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = MultinomialNB(alpha=best_params['alpha'])

    # Log the best model with MLflow
    log_mlflow("MultinomialNB", best_model, X_train, X_test, y_train, y_test)
    
    print(f"\nBest Optuna parameters: {best_params}")
    print(f"Best trial accuracy: {study.best_value:.4f}")

# Run the experiment for Multinomial Naive Bayes
run_optuna_experiment()